# This notebook serves to interact with FABRIC and DYNAMOS. 
- apply helm charts
- test FABRIC library (utilities)

## FABlib API References Examples

- [fablib.show_config](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.show_config)
- [fablib.list_sites](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_sites)
- [fablib.list_hosts](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.list_hosts)
- [fablib.new_slice](https://fabric-fablib.readthedocs.io/en/latest/fablib.html#fabrictestbed_extensions.fablib.fablib.FablibManager.new_slice)
- [slice.add_node](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.add_node)
- [slice.submit](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.submit)
- [slice.get_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.get_nodes)
- [slice.list_nodes](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.list_nodesß)
- [slice.show](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.show)
- [node.execute](https://fabric-fablib.readthedocs.io/en/latest/node.html#fabrictestbed_extensions.fablib.node.Node.execute)
- [slice.delete](https://fabric-fablib.readthedocs.io/en/latest/slice.html#fabrictestbed_extensions.fablib.slice.Slice.delete) 

In [1]:
%%time
import datetime
import json
import asyncio
from configuration import SLICE_NAME
from utils import upload_and_execute_file, upload_file, execute_file

from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

fablib.show_config();

User: apipilikas@gmail.com bastion key is valid!
Configuration is valid


Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
Token File,/home/fabric/.tokens.json
Project ID,49f65ad7-d8a2-4ab9-8ca0-ba777a2e0ea2
Bastion Host,bastion.fabric-testbed.net
Bastion Username,apipilikas_0000444352
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub
Slice Private Key File,/home/fabric/work/fabric_config/slice_key


CPU times: user 2.08 s, sys: 258 ms, total: 2.34 s
Wall time: 5.36 s


In [2]:
slice = fablib.get_slice(name=SLICE_NAME);
nodes = slice.get_nodes();
print(nodes)


[<fabrictestbed_extensions.fablib.node.Node object at 0x7ad56873ef90>, <fabrictestbed_extensions.fablib.node.Node object at 0x7ad57f4aa810>, <fabrictestbed_extensions.fablib.node.Node object at 0x7ad562c14690>, <fabrictestbed_extensions.fablib.node.Node object at 0x7ad562bcc6d0>, <fabrictestbed_extensions.fablib.node.Node object at 0x7ad562b83210>, <fabrictestbed_extensions.fablib.node.Node object at 0x7ad562b83750>, <fabrictestbed_extensions.fablib.node.Node object at 0x7ad57e78a9d0>]


In [3]:
# Print ssh information
try:
    # Get slice nodes
    for node in slice.get_nodes():
        print(f"Node: {node.get_name()}")
        # Get the original SSH command
        original_ssh_command = node.get_ssh_command()
        # Print SSH commands to get into the nodes
        print(f"  SSH Command from FABRIC: {original_ssh_command}")
        # Replace the file paths in the SSH command
        updated_ssh_command = original_ssh_command.replace(
            "/home/fabric/work/fabric_config/slice_key", "C:/Users/apipi/.ssh/slice_key"
        ).replace(
            "/home/fabric/work/fabric_config/ssh_config", "C:/Users/apipi/.ssh/fabric_ssh_config"
        )
        # Print the updated SSH command
        print(f"  SSH Command locally : {updated_ssh_command}")
    
except Exception as e:
    print(f"Fail: {e}")
    traceback.print_exc()

Node: control
  SSH Command from FABRIC: ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe28:8d36
  SSH Command locally : ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe28:8d36
Node: dynamos
  SSH Command from FABRIC: ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe76:1074
  SSH Command locally : ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe76:1074
Node: server
  SSH Command from FABRIC: ssh -i /home/fabric/work/fabric_config/slice_key -F /home/fabric/work/fabric_config/ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe49:ddf0
  SSH Command locally : ssh -i C:/Users/apipi/.ssh/slice_key -F C:/Users/apipi/.ssh/fabric_ssh_config ubuntu@2001:610:2d0:fabc:f816:3eff:fe49:ddf0
Node: clientone
 

In [4]:
# get nodes and IPs 
def get_ip(node):
    interface = node.get_interface(network_name=f"Network-{node.get_site()}")
    return interface.get_ip_addr()

nodes_dict= dict()

for node in nodes[:]:
    ip = get_ip(node)
    name = node.get_name()
    nodes_dict[name] = {"ip": ip, "node": node}
    print(f"{name}: {ip}")

# print(nodes_dict)

control: 10.145.2.2
dynamos: 10.145.2.3
server: 10.145.2.4
clientone: 10.145.2.5
clienttwo: 10.145.2.6
clientthree: 10.145.2.7
surf: 10.145.2.8


In [5]:
main_node=nodes_dict['control']['node']
print(f"The main node for now on is [{main_node.get_name()}]")

The main node for now on is [control]


In [11]:
upload_file(main_node, "overriden_files/temp/prometheus-values.yaml", "/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/monitoring/prometheus-values.yaml")
upload_file(main_node, "overriden_files/temp/loki.yaml", "/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/monitoring/templates/loki.yaml")
main_node.execute("helm upgrade -i prometheus prometheus-community/kube-prometheus-stack -f /home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/monitoring/prometheus-values.yaml -n monitoring")

Uploading file from local path [overriden_files/temp/prometheus-values.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/monitoring/prometheus-values.yaml] ...
Uploading file from local path [overriden_files/temp/loki.yaml] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/fabric/charts/monitoring/templates/loki.yaml] ...
Release "prometheus" has been upgraded. Happy Helming!
NAME: prometheus
LAST DEPLOYED: Wed Apr 22 15:44:30 2026
NAMESPACE: monitoring
STATUS: deployed
REVISION: 2
TEST SUITE: None
NOTES:
kube-prometheus-stack has been installed. Check its status by running:
  kubectl --namespace monitoring get pods -l "release=prometheus"

Get Grafana 'admin' user password by running:

  kubectl --namespace monitoring get secrets prometheus-grafana -o jsonpath="{.data.admin-password}" | base64 -d ; echo

Access Grafana local instance:

  export POD_NAME=$(kubectl --namespace monitoring get pod -l "app.kubernetes.io/name=grafana,app

('Release "prometheus" has been upgraded. Happy Helming!\nNAME: prometheus\nLAST DEPLOYED: Wed Apr 22 15:44:30 2026\nNAMESPACE: monitoring\nSTATUS: deployed\nREVISION: 2\nTEST SUITE: None\nNOTES:\nkube-prometheus-stack has been installed. Check its status by running:\n  kubectl --namespace monitoring get pods -l "release=prometheus"\n\nGet Grafana \'admin\' user password by running:\n\n  kubectl --namespace monitoring get secrets prometheus-grafana -o jsonpath="{.data.admin-password}" | base64 -d ; echo\n\nAccess Grafana local instance:\n\n  export POD_NAME=$(kubectl --namespace monitoring get pod -l "app.kubernetes.io/name=grafana,app.kubernetes.io/instance=prometheus" -oname)\n  kubectl --namespace monitoring port-forward $POD_NAME 3000\n\nGet your grafana admin user password by running:\n\n  kubectl get secret --namespace monitoring -l app.kubernetes.io/component=admin-secret -o jsonpath="{.items[0].data.admin-password}" | base64 --decode ; echo\n\n\nVisit https://github.com/prometh

In [6]:
upload_file(main_node, "overriden_files/temp/forward_ports.sh", "/home/ubuntu/scattered-directive-energy-monitoring/scripts/forward_ports.sh")

Uploading file from local path [overriden_files/temp/forward_ports.sh] to remote path [/home/ubuntu/scattered-directive-energy-monitoring/scripts/forward_ports.sh] ...


In [7]:
main_node.execute("find /home/ubuntu/scattered-directive-energy-monitoring -type f -name '*.sh' -exec sed -i 's/\\r$//' {} + -exec chmod +x {} +")

('', '')